# Why LLM Observability Differs

**Phase 07** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-07/01-why-llm-observability-differs.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your team just shipped an AI-powered support assistant. It responds in under 200ms, returns HTTP 200 on every request, and your Datadog dashboard is green. The SRE team is satisfied.

Then a customer emails: "Your AI told me my subscription auto-renews on the 1st but it doesn't. I missed my cancellation window." You search your logs. You find the request: status=200, latency=187ms. Nothing else. You cannot reconstruct what prompt was sent, what the model replied, which model version served it, how many tokens it consumed, or whether the answer came from cache. You are debugging a production hallucination with no evidence.

This is the core gap: traditional observability tools were designed f...

## The Concept

### Traditional vs. LLM Observability Signals

Traditional application performance monitoring (APM) captures three core signals: latency, error rate, and throughput. These are sufficient when outputs are deterministic. For LLM services, they are necessary but not sufficient.

```
+---------------------------+----------------------------------------+
| TRADITIONAL APM           | LLM OBSERVABILITY (additional)         |
+---------------------------+----------------------------------------+
| HTTP status code          | model name + version                   |
| Response latency (ms)     | promp...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Why LLM Observability Differs -- Phase 07 Lesson 01
appliedaifromscratch.com

Demonstrates: minimal structured logging for LLM API calls.
Captures the 8 essential fields every LLM request log must include:
    model, prompt_version, input_tokens, output_tokens, cost_usd,
    latency_ms, cache_hit, error

Run:
    pip install anthropic
    export ANTHROPIC_API_KEY=sk-...
    python main.py
"""

import json
import os
import time
from dataclasses import asdict, dataclass
from typing import Optional

import anthropic


# ---------------------------------------------------------------------------
# PRICING: token costs per million tokens (as of 2026)
# Update this table when Anthropic releases new pricing.
# ---------------------------------------------------------------------------

MODEL_COSTS: dict[str, dict[str, float]] = {
    "claude-3-5-haiku-20241022": {
        "input_per_m": 0.80,
        "output_per_m": 4.00,
        "cache_read_per_m": 0.08,
    },
    "claude-3-5-sonnet-20241022": {
        "input_per_m": 3.00,
        "output_per_m": 15.00,
        "cache_read_per_m": 0.30,
    },
}

### DATA MODEL: the 8 required fields

In [ ]:
@dataclass
class LLMLogRecord:
    """
    The 8 essential fields every LLM request log must capture.

    Why each field matters:
    - model: detect model drift when provider silently updates weights
    - prompt_version: bisect prompt regressions to the exact template change
    - input_tokens: track context window usage; alert on unexpectedly long prompts
    - output_tokens: track generation cost; alert on runaway completions
    - cost_usd: budget alerts; per-user cost attribution
    - latency_ms: SLA tracking; P95/P99 alerting
    - cache_hit: measure prompt cache effectiveness; optimize cache hit rate
    - error: classify failures (auth, rate limit, timeout, context length)
    """

    model: str
    prompt_version: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    latency_ms: float
    cache_hit: bool
    error: Optional[str]  # None on success; exception class name on failure

### COST CALCULATION

In [ ]:
def calculate_cost(
    model: str,
    input_tokens: int,
    output_tokens: int,
    cache_read_tokens: int = 0,
) -> float:
    """Calculate USD cost for an API call from token counts."""
    pricing = MODEL_COSTS.get(model, MODEL_COSTS["claude-3-5-haiku-20241022"])
    input_cost = (input_tokens / 1_000_000) * pricing["input_per_m"]
    output_cost = (output_tokens / 1_000_000) * pricing["output_per_m"]
    cache_cost = (cache_read_tokens / 1_000_000) * pricing["cache_read_per_m"]
    return round(input_cost + output_cost + cache_cost, 8)

### LOGGER

In [ ]:
class LLMLogger:
    """
    Minimal structured logger for LLM API calls.

    Wraps Anthropic API calls and emits a JSON log record per request.
    Records are written to stdout (for log aggregators) and optionally to
    a JSONL file (for local analysis).

    Usage:
        logger = LLMLogger(output_file="llm_requests.jsonl")
        text, record = logger.call("What is RAG?", prompt_version="support-v1")
    """

    def __init__(self, output_file: Optional[str] = None):
        self.client = anthropic.Anthropic()
        self.output_file = output_file

    def call(
        self,
        prompt: str,
        prompt_version: str,
        model: str = "claude-3-5-haiku-20241022",
        system: str = "You are a helpful assistant.",
        max_tokens: int = 512,
    ) -> tuple[str, LLMLogRecord]:
        """
        Call the Anthropic API and return (response_text, log_record).

        Always emits a structured log record, even on failure.
        On failure: error field is set, token counts are 0, response_text is "".
        """
        start = time.monotonic()
        error: Optional[str] = None
        response_text = ""
        input_tokens = 0
        output_tokens = 0
        cache_read_tokens = 0
        cache_hit = False

        try:
            response = self.client.messages.create(
                model=model,
                max_tokens=max_tokens,
                system=system,
                messages=[{"role": "user", "content": prompt}],
            )
            response_text = response.content[0].text
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens
            # Prompt cache: present when Anthropic returns cached tokens
            cache_read_tokens = (
                getattr(response.usage, "cache_read_input_tokens", 0) or 0
            )
            cache_hit = cache_read_tokens > 0

        except anthropic.APIError as exc:
            error = type(exc).__name__

        latency_ms = (time.monotonic() - start) * 1000

        record = LLMLogRecord(
            model=model,
            prompt_version=prompt_version,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            cost_usd=calculate_cost(model, input_tokens, output_tokens, cache_read_tokens),
            latency_ms=round(latency_ms, 2),
            cache_hit=cache_hit,
            error=error,
        )

        self._emit(record)
        return response_text, record

    def _emit(self, record: LLMLogRecord) -> None:
        """Write the log record as a JSON line to stdout and optionally to file."""
        line = json.dumps(asdict(record))
        print(line)
        if self.output_file:
            with open(self.output_file, "a") as f:
                f.write(line + "\n")

### SCHEMA VALIDATION

In [ ]:
def validate_records(path: str) -> None:
    """
    Read a JSONL log file and verify all 8 required fields are present.
    Run this after capturing a batch of requests to confirm logging integrity.
    """
    required_fields = [
        "model",
        "prompt_version",
        "input_tokens",
        "output_tokens",
        "cost_usd",
        "latency_ms",
        "cache_hit",
        "error",
    ]

    with open(path) as f:
        records = [json.loads(line) for line in f if line.strip()]

    for i, record in enumerate(records):
        for field in required_fields:
            assert field in record, f"Record {i} missing field: {field}"

    error_records = [r for r in records if r["error"] is not None]
    for r in error_records:
        assert r["input_tokens"] == 0, "Token counts must be 0 on API failure"
        assert r["cost_usd"] == 0.0, "Cost must be 0 on API failure"

    print(f"Validation passed: {len(records)} records, {len(error_records)} errors captured")

### DEMO

In [ ]:
def main() -> None:
    log_path = "llm_requests.jsonl"

    # Clear any previous run
    if os.path.exists(log_path):
        os.remove(log_path)

    logger = LLMLogger(output_file=log_path)

    print("=== LLM Structured Logger Demo ===\n")

    # Simulate requests from two different prompt templates
    requests = [
        ("What is the capital of France?", "general-qa-v1"),
        ("Summarize quantum computing in one sentence.", "summary-v2"),
        ("List three debugging strategies for distributed systems.", "support-v3"),
    ]

    for prompt, version in requests:
        text, record = logger.call(prompt=prompt, prompt_version=version)
        print(f"  Response: {text[:80].strip()}...")
        print(
            f"  Cost: ${record.cost_usd:.6f} | "
            f"Tokens: {record.input_tokens}in / {record.output_tokens}out | "
            f"Latency: {record.latency_ms:.0f}ms"
        )
        print()

    # Validate the captured log file
    print("=== Validating log file ===")
    validate_records(log_path)
    print(f"Log written to: {log_path}")
    print("\nKey insight: every record carries model + prompt_version.")
    print("When a prompt regression happens, you can filter by prompt_version")
    print("to find exactly when quality changed -- no guessing required.")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What does this situation reveal about your current observability setup?**

- A. Datadog is misconfigured and not receiving the correct metrics
- B. The HTTP layer cannot detect semantic failures -- you need LLM-specific logging
- C. A 0% error rate means the AI answer was correct; the customer is mistaken
- D. Latency metrics would have been higher if the model produced a wrong answer

**2. Which LLM-specific failure mode does this represent, and what field in your log would have caught it earliest?**

- A. Model drift; the model field would show the provider changed the model
- B. Runaway costs; the cost_usd field per request would show the per-request cost tripling
- C. Cascading tool failure; tool call logs would show repeated retries
- D. Prompt regression; the error field would show increased failure rates

**3. Which of the 8 essential log fields is most critical for diagnosing this issue, and why?**

- A. latency_ms -- slower responses correlate with lower quality outputs
- B. cache_hit -- the new template may be bypassing the cache and serving stale responses
- C. prompt_version -- you can filter logs by 'support-v3' to isolate when quality changed
- D. error -- the new template must be causing silent API errors

**4. What does this cache_hit field enable in production?**

- A. It confirms the model used cached model weights, which improves accuracy
- B. It lets you measure prompt cache hit rate and calculate how much you saved vs. full input token pricing
- C. It indicates the response was served from a CDN and did not reach the model
- D. It shows the user's previous request was identical, so no new inference was needed

**5. What is the main risk of this approach at production scale?**

- A. The Anthropic response object does not support serialization to JSON
- B. The raw response contains sensitive content that should not be logged
- C. Unstructured logs are expensive to query, schema changes in the SDK break your log parsers, and you cannot alert on fields you have not pre-defined
- D. Logging the full response object will cause latency to increase by more than 100ms

**6. What limitation of flat log records does this scenario expose?**

- A. Flat log records cannot capture tool call results, making it impossible to see what data each tool returned
- B. Three separate records have no shared identifier linking them to the same user request, and no way to show the causal sequence of tool calls
- C. Logging multiple records per user request violates data retention policies
- D. The cost_usd field will be inaccurate when summed across multiple records

**7. What does this scenario illustrate about the error field in LLM logging?**

- A. The error field is redundant if you have an API gateway with retry logic
- B. LLM-specific errors like rate limits are hidden by retry logic at the transport layer, but logging them reveals capacity and cost problems before they become outages
- C. RateLimitError in the log means you should switch to a different model provider
- D. The error field only captures network errors, not API-level errors

**8. What production action does the cost_usd field now enable that was not possible before?**

- A. Blocking high-cost users automatically to stay within budget
- B. Per-user cost attribution, usage-based billing, per-user rate limiting, and identifying prompt patterns that drive high token usage
- C. Negotiating a lower per-token price with Anthropic based on usage volume
- D. Switching to a cheaper model automatically for high-cost sessions

_Answers are in `checks.json` in the lesson directory._